# Arctic Ocean

Arctic Ocean hydrography

In [ ]:
# parameters for the cmorized data
caseid          = ''
input_root      = ''
work_root       = ''
www_root        = ''
year_start      = []
year_end        = []

In [ ]:
import yaml
import xarray as xr
import sys
import os
import numpy as np
import xarray as xry
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from pathlib import Path
import matplotlib as mpl
import cartopy.crs as ccrs
import matplotlib.path as mpath
import xesmf as xe
from pathlib import Path
with open("../settings.yml", "r") as f:
    cfg = yaml.safe_load(f)

sys.path.append(cfg["blomster"]["source_root"])
from utils.plots import plot_map2d
from utils import get_grid
from utils.plots import plot_arctic_maps
from utils.analysis import mean_at_depth,load_ocean_grid,deep_ocean_mask,shared_quantile_levels,symmetric_quantile_levels
# with xr.open_dataset(cfg['grids']['tnx1v4']['grid_file']) as ds:
#     lat = ds['plat']
#     lon = ds['plon']

# with xr.open_dataset(cfg['climos']['blom']['hm']['MAR']) as ds:
#     mld04 = ds['mldb04'].isel(time=0)

# plot_map2d(lon, lat, mld04, proj='NearsidePerspective', cmap='viridis')

In [ ]:
# delete before running
# caseid = "n1850GaxgGHG.LM.nor30b25.528.20260928"
# input_root = "/nird/datalake/NS9560K/noresm3/cases"
# work_root = "/scratch/yanchun/diagnostics/blomster"
# www_root = "/nird/datalake/NS9015K/www/diagnostics/blomster/nililiu"
# year_start = 1596
# year_end = 1600

## Hovmoller diagrams

### 1.values change with time

In [ ]:
time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
grid_file = cfg['grids']['tnx1v4']['grid_file']
years = range(year_start, year_end+1)
hist_dir = Path(f"{input_root}/{caseid}/ocn/hist")
files = [hist_dir / f"{caseid}.blom.hy.{yr}.nc" for yr in years]

max_depth = 1500
with xr.open_dataset(grid_file) as grid:
    arctic_mask = ((grid["pmask"] > 0)& (grid["plat"] >= 65)& (grid["pdepth"] > 500))
    weights = (grid["parea"].where(arctic_mask, 0).fillna(0).load())
    
temp_profiles = []
salt_profiles = []
for i, file in enumerate(files, 1):
    with xr.open_dataset(file, decode_times=time_coder) as ds:
        temp_one = (ds["templvl"].sel(depth=slice(0, 4000)).weighted(weights).mean(("y", "x"), skipna=True).load())
        salt_one = (ds["salnlvl"].sel(depth=slice(0, 4000)).weighted(weights).mean(("y", "x"), skipna=True).load())
    temp_profiles.append(temp_one)
    salt_profiles.append(salt_one)
temp_time_depth = xr.concat(temp_profiles, dim="time")
salt_time_depth = xr.concat(salt_profiles, dim="time")

temp_time_levels = shared_quantile_levels([temp_time_depth],nlevels=21)
salt_time_levels = shared_quantile_levels([salt_time_depth],nlevels=21)

In [ ]:
plot_years = temp_time_depth["time"].dt.year.values
fig, ax = plt.subplots(figsize=(9, 6))
mesh = ax.pcolormesh(
    plot_years,
    temp_time_depth["depth"] / 1000,
    temp_time_depth.transpose("depth", "time"),
    cmap="plasma",
    vmin=temp_time_levels[0],
    vmax=temp_time_levels[-1],
    shading="auto",
)

colorbar = fig.colorbar(mesh, ax=ax, pad=0.02)
colorbar.set_label("Temperature [°C]")

ax.set_xlabel("Model year",fontsize=12)
ax.set_ylabel("Depth [km]",fontsize=12)
ax.set_title("Arctic Ocean temperature",fontsize=14)
ax.set_ylim(max_depth / 1000, 0)

fig.suptitle(f"{caseid}\nArea-weighted mean north of 65°N",fontsize=12,y=0.98)

fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
mesh = ax.pcolormesh(
    plot_years,
    salt_time_depth["depth"] / 1000,
    salt_time_depth.transpose("depth", "time"),
    cmap="plasma",
    vmin=salt_time_levels[0],
    vmax=salt_time_levels[-1],
    shading="auto",
)

colorbar = fig.colorbar(mesh, ax=ax, pad=0.02)
colorbar.set_label("Salnility [g/kg]")

ax.set_xlabel("Model year",fontsize=12)
ax.set_ylabel("Depth [km]",fontsize=12)
ax.set_title("Arctic Ocean Salnility",fontsize=14)
ax.set_ylim(max_depth / 1000, 0)
fig.suptitle(f"{caseid}\nArea-weighted mean north of 65°N",fontsize=12,y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

### 2. anomaly change with time

In [ ]:
temp_anomaly = (temp_time_depth- temp_time_depth.mean("time", skipna=True))
salt_anomaly = (salt_time_depth- salt_time_depth.mean("time", skipna=True))
temp_ano_levels = symmetric_quantile_levels(temp_anomaly,nlevels=21)
salt_ano_levels = symmetric_quantile_levels(salt_anomaly,nlevels=21)
fig, ax = plt.subplots(figsize=(9, 6))
mesh = ax.pcolormesh(
    plot_years,
    temp_anomaly["depth"] / 1000,
    temp_anomaly.transpose("depth", "time"),
    cmap="RdBu_r",
    vmin=temp_ano_levels[0],
    vmax=temp_ano_levels[-1],
    shading="auto",
)

colorbar = fig.colorbar(mesh, ax=ax, pad=0.02)
colorbar.set_label("Temperature [°C]")

ax.set_xlabel("Model year",fontsize=12)
ax.set_ylabel("Depth [km]",fontsize=12)
ax.set_title("Arctic Ocean temperature anomaly",fontsize=14)
ax.set_ylim(max_depth / 1000, 0)

fig.suptitle(f"{caseid}\nArea-weighted mean north of 65°N",fontsize=12,y=0.98)

fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))
mesh = ax.pcolormesh(
    plot_years,
    salt_anomaly["depth"] / 1000,
    salt_anomaly.transpose("depth", "time"),
    cmap="RdBu_r",
    vmin=salt_ano_levels[0],
    vmax=salt_ano_levels[-1],
    shading="auto",
)

colorbar = fig.colorbar(mesh, ax=ax, pad=0.02)
colorbar.set_label("Salnility [g/kg]")

ax.set_xlabel("Model year",fontsize=12)
ax.set_ylabel("Depth [km]",fontsize=12)
ax.set_title("Arctic Ocean sanility anomaly",fontsize=14)
ax.set_ylim(max_depth / 1000, 0)

fig.suptitle(f"{caseid}\nArea-weighted mean north of 65°N",fontsize=12,y=0.98)

fig.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()

## Vertical 
1. temp profile with observation
2. salt profile with observation

In [ ]:
time_coder = xr.coders.CFDatetimeCoder(use_cftime=True)
grid_file = cfg['grids']['tnx1v4']['grid_file']
years = range(year_start, year_end+1)
hist_dir = Path(f"{input_root}/{caseid}/ocn/hist")
files = [hist_dir / f"{caseid}.blom.hy.{yr}.nc" for yr in years]

phc_file = cfg['datasets']['PHC3.0']['ANNUAL_file']
with xr.open_dataset(grid_file) as grid:
    arctic_mask = ((grid["pmask"] > 0) & (grid["plat"] >= 65) & (grid["pdepth"] > 500))
    weights = (grid["parea"].where(arctic_mask, 0).fillna(0).load())

temp_profiles = []
salt_profiles = []

for i, file in enumerate(files, 1):
    with xr.open_dataset(file, decode_times=time_coder) as ds:
        temp_one = (ds["templvl"].sel(depth=slice(0, 4000)).weighted(weights).mean(("y", "x"), skipna=True).load())
        salt_one = (ds["salnlvl"].sel(depth=slice(0, 4000)).weighted(weights).mean(("y", "x"), skipna=True).load())
    temp_profiles.append(temp_one)
    salt_profiles.append(salt_one)

temp_profile = (xr.concat(temp_profiles, dim="time").mean("time", skipna=True))
salt_profile = (xr.concat(salt_profiles, dim="time").mean("time", skipna=True))

with xr.open_dataset(phc_file,decode_times=time_coder,) as phc:
    phc_weights = np.cos(np.deg2rad(phc["lat"]))
    phc_weights = phc_weights.where(phc["lat"] >= 65, 0)
    phc_temp_profile = (phc["temp"].sel(depth=slice(0, 4000)).weighted(phc_weights).mean(("lat", "lon"), skipna=True).load())
    phc_salt_profile = (phc["salt"].sel(depth=slice(0, 4000)).weighted(phc_weights).mean(("lat", "lon"), skipna=True).load())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6.5))

ax.plot(temp_profile,temp_profile["depth"] / 1000,color="tab:red",linewidth=2.2,label="NorESM3")

ax.plot(phc_temp_profile,phc_temp_profile["depth"] / 1000,color="black",linestyle="-",linewidth=2,label="PHC3.0")

ax.set_xlabel("Temperature [°C]",fontsize=12)
ax.set_ylabel("Depth [km]",fontsize=12)
ax.set_title("Arctic Ocean temperature profile",fontsize=14)

ax.set_ylim(4, 0)
ax.grid(True, alpha=0.3)
ax.legend()
fig.suptitle(f"{caseid}\nYears {year_start}–{year_end}",fontsize=12,y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6.5))

ax.plot(salt_profile,salt_profile["depth"] / 1000,color="tab:blue",linewidth=2.2,label="NorESM3",)

ax.plot(phc_salt_profile,phc_salt_profile["depth"] / 1000,color="black",linestyle="-",linewidth=2,label="PHC3.0")

ax.set_xlabel("Salinity [g kg$^{-1}$]",fontsize=12)
ax.set_ylabel("Depth [km]",fontsize=12)
ax.set_title("Arctic Ocean salinity profile",fontsize=14)

ax.set_ylim(4, 0)
ax.grid(True, alpha=0.3)
ax.legend()

fig.suptitle(f"{caseid}\nYears {year_start}–{year_end}",fontsize=12,y=0.98)
fig.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

## Climatology maps
1. surface T & S
2. 400m T & S
3. 1000m T & S
4. 2000m T & S

In [ ]:
hist_dir = Path(f"{input_root}/{caseid}/ocn/hist")
years = range(year_start, year_end + 1)
files = [hist_dir / f"{caseid}.blom.hy.{year}.nc" for year in years]

In [ ]:
# Climatology maps
depths = [0,400,1000,2000]
grid_file = cfg['grids']['tnx1v4']['grid_file']
plat, plon, pclat, pclon , pmask = get_grid.load(grid_file)
phc_file = cfg['datasets']['PHC3.0']['ANNUAL_file']
time_coder=xr.coders.CFDatetimeCoder(use_cftime=True)

temp_model_arctic={}
salt_model_arctic={}
phc_temp_on_model={}
phc_salt_on_model={}
temp_bias={}
salt_bias={}
temp_levels={}
salt_levels={}
temp_bias_levels={}
salt_bias_levels={}
for depth in depths:
    temp_model=mean_at_depth("templvl",files,depth)
    salt_model=mean_at_depth("salnlvl",files,depth)
    ydim, xdim = temp_model.dims[-2:]
    # Arctic
    arctic = xr.DataArray((np.asarray(pmask) > 0) & (np.asarray(plat) >= 60),dims=(ydim, xdim))
    temp_model_arctic[depth] = temp_model.where(arctic)
    salt_model_arctic[depth] = salt_model.where(arctic)
    # interp
    with xr.open_dataset(phc_file, decode_times=time_coder) as ds:
        phc_temp = ds["temp"].interp(depth=depth).load()
        phc_salt = ds["salt"].interp(depth=depth).load()
    # create target grid
    model_grid = xr.Dataset(
        {
            "lon": ((ydim, xdim),np.mod(np.asarray(plon), 360),),
            "lat": ((ydim, xdim),np.asarray(plat),),
        }
    )
    # PHC
    phc_grid = xr.Dataset(coords={"lon": phc_temp["lon"],"lat": phc_temp["lat"],})
    # PHC → model grid
    regridder = xe.Regridder(phc_grid,model_grid,method="bilinear",periodic=True,)
    # skipnan
    phc_temp_on_model[depth] = regridder(phc_temp,skipna=True,na_thres=1.0,).where(arctic)
    phc_salt_on_model[depth] = regridder(phc_salt,skipna=True,na_thres=1.0,).where(arctic)
    # Bias
    temp_bias[depth] = temp_model_arctic[depth]- phc_temp_on_model[depth]
    salt_bias[depth] = salt_model_arctic[depth]- phc_salt_on_model[depth]
    temp_levels[depth] = shared_quantile_levels([temp_model_arctic[depth]],nlevels=21)
    salt_levels[depth] = shared_quantile_levels([salt_model_arctic[depth]],nlevels=21)
    temp_bias_levels[depth] = symmetric_quantile_levels(temp_bias[depth],nlevels=21)
    salt_bias_levels[depth] =symmetric_quantile_levels(salt_bias[depth],nlevels=21)

def plot_model_depth(depth):
    model_settings = [
        (
            temp_model_arctic[depth],
            temp_levels[depth],
            "plasma",
            "Temperature [°C]",
        ),
        (
            salt_model_arctic[depth],
            salt_levels[depth],
            "viridis",
            "Salinity [g kg$^{-1}$]",
        ),
    ]

    model_titles = [
        f"Model Arctic Ocean temperature at {depth} m",
        f"Model Arctic Ocean salinity at {depth} m",
    ]

    fig, axes = plot_arctic_maps(
        lon=plon,
        lat=plat,
        settings=model_settings,
        titles=model_titles,
        suptitle=f"{caseid}\nYears {year_start}–{year_end},depth={depth} m",
        layout='horizontal'
    )

    plt.show()

In [ ]:
plot_model_depth(0)

In [ ]:
plot_model_depth(400)

In [ ]:
plot_model_depth(1000)

In [ ]:
plot_model_depth(2000)

## Bias maps
model_result - PHC3.0_results
1. surface
2. 400m
3. 1000m
4. 2000m

In [ ]:
def plot_bias_depth(depth):
    bias_settings = [
        (
            temp_bias[depth],
            temp_bias_levels[depth],
            "RdBu_r",
            "Temperature bias [°C]",
        ),
        (
            salt_bias[depth],
            salt_bias_levels[depth],
            "RdBu_r",
            "Salinity bias",
        ),
    ]

    bias_titles = [
        f"Temperature bias at {depth} m",
        f"Salinity bias at {depth} m",
    ]

    fig, axes = plot_arctic_maps(
        lon=plon,
        lat=plat,
        settings=bias_settings,
        titles=bias_titles,
        suptitle=(
            f"{caseid}\n"
            f"Years {year_start}–{year_end}, depth = {depth} m"
        ),
        layout='horizontal'
    )

    plt.show()
    

In [ ]:
plot_bias_depth(0)

In [ ]:
plot_bias_depth(400)

In [ ]:
plot_bias_depth(1000)

In [ ]:
plot_bias_depth(2000)

## Transects
Greenland to Svalbard profile
1. temperature section
2. salinity section
3. normal velocty section
4. volume transport
5. heat transport
6. freshwater transport

In [ ]:
# Transects across the Fram Strait
# recognize diffirent basins
